### Pulso interactivo 
Este código te permite generar un pulso interactivo  de potencial de acción con los siguientes parámetros modificables :
- Temperatura 
- Duración del pulso  : dt
- Valores de A y B
- Intensidad del pulso

Si se quieren modificar los rangos de valores posibles de variación se hace en la línea :
tc=pw.FloatSlider(min=6, max=35, step=1, value=6, continuous_update=False)
Y mover los valores de min, max o el paso.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.integrate as integrate
import ipywidgets as pw

# 1. Parámetros globales fisiológicos
Cm = 1.0
Vk, VNa, VL = -77.0, 50.0, -54.387

# Conductancias modificadas 
def conductances(A, B, T):
    fact = A * (1 + B * (T - 6.3))
    gk = 36.0 * fact
    gNa = 120.0 * fact #Factor eta multiplicando a las conductancias
    gL = 0.3*fact
    return gk, gNa, gL

# 2. Factor Térmico
def temp(T): 
    return 3.0 ** ((T - 6.3) / 10.0)

# 3. Funciones de Transición con valores experimentales de HH
def alpha_n(V_rel, phi): return phi * (0.01 * (10.0 - V_rel) / (np.exp(1.0 - 0.1 * V_rel) - 1.0 + 1e-9))
def beta_n(V_rel, phi): return phi * (0.125 * np.exp(-V_rel / 80.0))
def alpha_m(V_rel, phi): return phi * (0.1 * (25.0 - V_rel) / (np.exp(2.5 - 0.1 * V_rel) - 1.0 + 1e-9))
def beta_m(V_rel, phi): return phi * (4.0 * np.exp(-V_rel / 18.0))
def alpha_h(V_rel, phi): return phi * (0.07 * np.exp(-V_rel / 20.0))
def beta_h(V_rel, phi): return phi * (1.0 / (np.exp(3.0 - 0.1 * V_rel) + 1.0))
#Valores en el equlibrio t en infinito 
def n_inf(V_rel, phi): return alpha_n(V_rel, phi) / (alpha_n(V_rel, phi) + beta_n(V_rel, phi))
def m_inf(V_rel, phi): return alpha_m(V_rel, phi) / (alpha_m(V_rel, phi) + beta_m(V_rel, phi))
def h_inf(V_rel, phi): return alpha_h(V_rel, phi) / (alpha_h(V_rel, phi) + beta_h(V_rel, phi))

# 4. Estímulo Eléctrico
def Id_tvar(t, pul, delt):
    if 10.0 <= t <= 10.0 + delt:#Se suma el dt para ajustar la duración del pulso 
        return pul
    else:
        return 0.0

# 5. Integrador Principal
def CalcDeriv(y, t, phi, pul, V_rest, dt_pulso, gk, gNa, gL):
    dy = np.zeros(4)
    
    Vm, n, m, h = y
    #Corrientes de paso en cada canal 
    IK = gk * (n**4) * (Vm - Vk) 
    INa = gNa * (m**3) * h * (Vm - VNa)
    IL = gL * (Vm - VL)
    V_rel = Vm - V_rest
    # Estas se integraran 
    dy[0] = (Id_tvar(t, pul, dt_pulso) - IK - INa - IL) / Cm #Ecuación principal de la memabran 
    #Ecuaciones para las variables aleatorias 
    dy[1] = alpha_n(V_rel, phi) * (1.0 - n) - beta_n(V_rel, phi) * n 
    dy[2] = alpha_m(V_rel, phi) * (1.0 - m) - beta_m(V_rel, phi) * m
    dy[3] = alpha_h(V_rel, phi) * (1.0 - h) - beta_h(V_rel, phi) * h
    return dy   
    
# Generar pulso
def pulso(tc, pul, dp, A, B):
    # Se calcula el factor phi para insluir efectos de Temperatura 
    phi = temp(tc)
    V_rest = -65.0
    Vm_0 = -65.0
    V_rel_0 = Vm_0 - V_rest
    
    # Se calculan las conductancias dependientes de la temperatura 
    gk, gNa, gL = conductances(A, B, tc)
    
    Y0 = np.array([Vm_0, n_inf(V_rel_0, phi), m_inf(V_rel_0, phi), h_inf(V_rel_0, phi)]) #Valores inciales
    T1 = np.linspace(0, 50, 10000) 
    #hmax controla la resolución  ya que establece el paso de integración máximo permitido
    Vy = integrate.odeint(CalcDeriv, Y0, T1, args=(phi, pul, V_rest, dp, gk, gNa, gL), hmax=0.05) #Integración #0.05 en el caso interactivo 
    Sol1 = Vy.T
    return Sol1

def pulso_interactivo(tc, pul, dp, A, B):
    resp = pulso(tc, pul, dp, A, B) #Calculamos el pulso 
    T1 = np.linspace(0, 50, 10000) #Pasos de tiempo
    plt.figure(figsize=(10, 5))
    plt.plot(T1, resp[0], label=f'Duración del pulso {dp} ms', color='red', linewidth=1.5)
    plt.vlines(dp+10, -100, 70, color="black", linestyle="--") #Línea para marcar la duración del pulso 
    plt.title(f'Potencial de Acción: T = {tc}°C | I = {pul} $\mu A/cm^2$ | A = {A}')
    plt.xlabel('Tiempo (ms)')
    plt.ylabel('Voltaje (mV)')
    plt.ylim(-90, 60)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.savefig("Interactive.png")
    plt.show()
    

<>:83: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<>:83: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
C:\Users\barto\AppData\Local\Temp\ipykernel_11448\1195891887.py:83: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
  plt.title(f'Potencial de Acción: T = {tc}°C | I = {pul} $\mu A/cm^2$ | A = {A}')


In [2]:
interactive_plot = pw.interact(
    pulso_interactivo, 
    tc=pw.FloatSlider(min=6, max=35, step=1, value=6, continuous_update=False),
    pul=pw.FloatSlider(min=0, max=250, step=1, value=20, continuous_update=False),
    dp=pw.FloatSlider(min=0.1, max=10, step=0.1, value=1.0, continuous_update=False),
    A=pw.FloatSlider(min=1, max=4, step=1, value=1, continuous_update=False),
    B=pw.FloatSlider(min=0.0, max=0.063, step=0.063, value=0.063, continuous_update=False)
)

interactive(children=(FloatSlider(value=6.0, continuous_update=False, description='tc', max=35.0, min=6.0, ste…